# Phase 07 — Exploratory Data Analysis (EDA)
## CodeMentor-LLM
Exploring the cleaned CodeAlpaca-20K dataset
to understand data distribution before training.

## Analysis Steps
1. Token length distribution
2. Response length distribution
3. Top 20 most common instruction keywords
4. Topic balance analysis
5. Summary statistics

# libraries

In [ ]:
!pip install -q transformers==4.49.0 datasets==3.3.2 pandas==2.2.3 matplotlib==3.10.1 seaborn==0.13.2

# Login and load dataset

In [ ]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

# Login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Load cleaned dataset
print("\nLoading cleaned dataset...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-cleaned")
df = pd.DataFrame(dataset["train"])

print(f"Dataset loaded successfully")
print(f"Total samples: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Calculate token lengths

In [ ]:
from transformers import AutoTokenizer

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

# Calculate token lengths
print("Calculating token lengths...")
df["token_length"] = df["text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

# Summary stats
print(f"\nToken Length Statistics:")
print(f"Min    : {df['token_length'].min()}")
print(f"Max    : {df['token_length'].max()}")
print(f"Mean   : {df['token_length'].mean():.2f}")
print(f"Median : {df['token_length'].median()}")
print(f"Std    : {df['token_length'].std():.2f}")

# Token length distribution plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("plots", exist_ok=True)

# Plot token length distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df["token_length"], bins=50, color="#4C72B0")
plt.title("Token Length Distribution")
plt.xlabel("Token Length")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
sns.boxplot(y=df["token_length"], color="#4C72B0")
plt.title("Token Length Boxplot")
plt.ylabel("Token Length")

plt.tight_layout()
plt.savefig("plots/token_length_distribution.png", dpi=150)
plt.show()
print("Plot saved")

# Response length distribution

In [ ]:
# Extract assistant responses
def extract_response(text):
    if "<|start_header_id|>assistant<|end_header_id|>" in text:
        response = text.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        response = response.replace("<|eot_id|>", "").strip()
        return response
    return ""

df["response"] = df["text"].apply(extract_response)
df["response_word_count"] = df["response"].apply(lambda x: len(x.split()))

# Plot response length distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df["response_word_count"], bins=50, color="#DD8452")
plt.title("Response Word Count Distribution")
plt.xlabel("Word Count")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
sns.boxplot(y=df["response_word_count"], color="#DD8452")
plt.title("Response Word Count Boxplot")
plt.ylabel("Word Count")

plt.tight_layout()
plt.savefig("plots/response_length_distribution.png", dpi=150)
plt.show()
print(f"\nResponse Word Count Stats:")
print(f"Min    : {df['response_word_count'].min()}")
print(f"Max    : {df['response_word_count'].max()}")
print(f"Mean   : {df['response_word_count'].mean():.2f}")
print(f"Median : {df['response_word_count'].median()}")

# Top 20 most common instruction keywords

In [ ]:
from collections import Counter
import re

# Extract instructions from text
def extract_instruction(text):
    if "<|start_header_id|>user<|end_header_id|>" in text:
        instruction = text.split("<|start_header_id|>user<|end_header_id|>")[-1]
        instruction = instruction.split("<|eot_id|>")[0].strip()
        return instruction
    return ""

df["instruction"] = df["text"].apply(extract_instruction)

# Get top 20 keywords
all_words = []
for instruction in df["instruction"]:
    words = re.findall(r'\b[a-zA-Z]+\b', instruction.lower())
    all_words.extend(words)

# Remove stop words
stop_words = {"a", "an", "the", "in", "to", "of", "and", "or",
              "is", "for", "with", "that", "it", "on", "at",
              "this", "be", "by", "from", "as", "are", "you",
              "given", "using", "which", "can", "will", "how"}

filtered_words = [w for w in all_words if w not in stop_words and len(w) > 2]
word_counts = Counter(filtered_words).most_common(20)

# Plot
words, counts = zip(*word_counts)
plt.figure(figsize=(12, 6))
sns.barplot(x=list(counts), y=list(words), palette="viridis")
plt.title("Top 20 Most Common Instruction Keywords")
plt.xlabel("Count")
plt.ylabel("Keyword")
plt.tight_layout()
plt.savefig("plots/top_20_keywords.png", dpi=150)
plt.show()
print("Top 20 keywords:", word_counts)

# Topic balance analysis

In [ ]:
# Categorize instructions by topic
def categorize_instruction(text):
    text_lower = text.lower()
    if any(word in text_lower for word in ["debug", "error", "fix", "wrong", "issue", "bug"]):
        return "debug"
    elif any(word in text_lower for word in ["explain", "what is", "describe", "difference", "how does"]):
        return "explain"
    elif any(word in text_lower for word in ["write", "create", "build", "implement", "generate", "make"]):
        return "write_code"
    elif any(word in text_lower for word in ["optimize", "improve", "refactor", "better"]):
        return "optimize"
    else:
        return "other"

df["topic"] = df["instruction"].apply(categorize_instruction)
topic_counts = df["topic"].value_counts()

# Plot
plt.figure(figsize=(10, 5))
sns.barplot(x=topic_counts.index, y=topic_counts.values, palette="Set2")
plt.title("Topic Balance Analysis")
plt.xlabel("Topic")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("plots/topic_balance.png", dpi=150)
plt.show()

print("Topic Distribution:")
for topic, count in topic_counts.items():
    print(f"  {topic:15} : {count:5} ({count/len(df)*100:.2f}%)")